# FRED Derived Subcategories

FRED's taxonomy bottoms out above browsable size: 206 categories hold 500 or more series
attached directly to them and the largest holds 53,187, so the nodes between a category
and its series do not exist upstream and have to be derived. This notebook examines the
result — not whether the pipeline ran, but whether what it produced is usable.

The code is `apps/agentic/db/fred_subcategory.py`; the entry point is
`clients/bin/fred_subcategories.py`; the specification is
`sefer/planning/yada-fred-subcategory-procedure.md`. Collection — how the series got here
in the first place — is `fred_meta_data.ipynb`.

**This notebook is read-only.** Every cell is a `SELECT`. Nothing here embeds, names,
clusters, or writes, so it costs nothing and cannot disturb a build in progress.

### What to look for

Three sections carry findings rather than confirmations:

- **§5** — the vocabulary bridge works, but **not under `websearch_to_tsquery`**. The
  operator that feels like the obvious choice returns zero rows for queries the layer
  answers well.
- **§6** — **16% of nodes share a name with a sibling.** The semantic scheme fails on
  *facet* categories, where titles differ by place rather than by measure. This is the
  same defect that excludes the regional half, and it is present in the half that was
  built.
- **§8** — what is deliberately absent, and what is a fault.

## 0. Setup

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath('../../..'))

from dotenv import load_dotenv
load_dotenv(os.path.abspath('../../../.env'), override=True)

from sqlalchemy import text

from apps.agentic.db.fred_catalogue import _engine
from apps.agentic.db import fred_subcategory as fs

engine = _engine()
with engine.connect() as c:
    print('alembic head in db:', c.execute(text('SELECT version_num FROM alembic_version')).scalar())
print('prompt version    :', fs.PROMPT_VERSION)
print('split version     :', fs.SPLIT_VERSION)
print('leaf cap / branch :', fs.LEAF_CAP, '/', fs.BRANCH_CAP)

alembic head in db: 0011
prompt version    : 2026-11-01.1
split version     : 2026-11-08.balanced
leaf cap / branch : 25 / 25


Both versions are stored on every row, and `pending_builds` treats a difference in
*either* as pending work. The names are model output and the shape is splitter output, so
a tree half-built under each is not a tree — and without `split_version` a rebuilt
splitter would leave the old shape reading as finished forever.

## 1. What the build produced

`status` separates two very different outcomes. **`ok`** means the category was split and
named. **`skipped`** means it was already under the cap — it holds 25 series or fewer, so
deriving nodes beneath it would add a click and no information.

`pending_builds` returns every category in scope, not just the over-cap ones, which is
why a run reports more categories than it names.

In [2]:
with engine.connect() as c:
    print(f"{'status':<9} {'cats':>5} {'series':>9} {'nodes':>7} {'leaves':>7} {'names':>7} {'cost':>9} {'fails':>6}")
    for r in c.execute(text('''
        SELECT status, count(*), sum(series_count), sum(nodes), sum(leaves),
               sum(llm_calls), sum(cost_usd), sum(naming_failures)
          FROM fred_subcategory_build GROUP BY status ORDER BY 2 DESC
    ''')):
        print(f'{r[0]:<9} {r[1]:>5} {r[2] or 0:>9,} {r[3] or 0:>7,} {r[4] or 0:>7,} '
              f'{r[5] or 0:>7,} {"$" + format(r[6] or 0, ".4f"):>9} {r[7] or 0:>6}')

    print()
    for label, q in [
        ('subcategory nodes', 'SELECT count(*) FROM fred_subcategory'),
        ('  of which leaves', 'SELECT count(*) FROM fred_subcategory WHERE is_leaf'),
        ('memberships      ', 'SELECT count(*) FROM fred_series_subcategory'),
        ('distinct series  ', 'SELECT count(DISTINCT series_id) FROM fred_series_subcategory'),
    ]:
        print(f'{label} {c.execute(text(q)).scalar():>9,}')

status     cats    series   nodes  leaves   names      cost  fails
ok          723   469,329  27,385  21,602  27,385   $4.8976     38
skipped     201     1,939       0       0       0   $0.0000      0

subcategory nodes    27,385
  of which leaves    21,602
memberships         469,329


distinct series     417,133


`memberships` exceeds `distinct series` because a series belonging to several FRED
categories is placed in a leaf under *each* of them. Clustering happens **within** a
category, never globally — cheaper, sharper names because the input is already
homogeneous, and it keeps FRED's own curation as the spine instead of replacing it with
tens of thousands of derived nodes and no human structure above them.

## 2. The invariants, checked in both directions

The requirement is that a node can be read by eye. That takes two ceilings, not one, and
a tree can satisfy the first completely while being unbrowsable at the top:

- **`leaf_cap_violations`** — no leaf holds more than 25 series.
- **`branch_cap_violations`** — no node has more than 25 children.

The second was documented but **not enforced** until 2026-11-08: the `k` ceiling read
`branch ** 2`, so a node could take 625 children, and 130 of the 723 non-regional
categories were over the threshold that triggers it. A leaf holding 1,000 series and a
node with 625 children are equally unusable; only one of them was being prevented.

Both are read back **from the database**, not inferred from what the build believed it
wrote.

In [3]:
leaf_bad = fs.leaf_cap_violations(engine)
branch_bad = fs.branch_cap_violations(engine)
unnamed = fs.unnamed_nodes(engine)

print(f'leaves over {fs.LEAF_CAP} series   : {len(leaf_bad)}')
print(f'nodes over {fs.BRANCH_CAP} children  : {len(branch_bad)}')
print(f'unnamed nodes          : {len(unnamed)}')

for category_id, subcategory_id, size in leaf_bad[:5]:
    print(f'  leaf {subcategory_id} in category {category_id}: {size} series')
for category_id, subcategory_id, kids in branch_bad[:5]:
    print(f'  node {subcategory_id} in category {category_id}: {kids} children')

leaves over 25 series   : 0
nodes over 25 children  : 0
unnamed nodes          : 38


A third check is stronger than either, because it cannot be satisfied by building a
well-shaped tree that quietly loses rows: **every membership in a built category must
land in exactly one leaf of that category.**

In [4]:
with engine.connect() as c:
    unplaced = c.execute(text('''
        SELECT count(*) FROM fred_series_category sc
          JOIN fred_subcategory_build b
            ON b.category_id = sc.category_id AND b.status = 'ok'
         WHERE NOT EXISTS (
            SELECT 1 FROM fred_series_subcategory ss
              JOIN fred_subcategory s USING (subcategory_id)
             WHERE ss.series_id = sc.series_id AND s.category_id = sc.category_id)
    ''')).scalar()
    counted = c.execute(text(
        'SELECT sum(series_count) FROM fred_subcategory WHERE is_leaf')).scalar()
    actual = c.execute(text('SELECT count(*) FROM fred_series_subcategory')).scalar()

print(f'memberships in a built category but in no leaf : {unplaced:,}')
print(f'sum of leaf series_count                       : {counted:,}')
print(f'actual membership rows                         : {actual:,}')
print(f'stored counts agree with reality               : {counted == actual}')

memberships in a built category but in no leaf : 0
sum of leaf series_count                       : 469,329
actual membership rows                         : 469,329
stored counts agree with reality               : True


The second pair matters separately: `series_count` is a stored denormalisation, so it can
drift from the membership rows it summarises. Every leaf-size claim in this notebook reads
that column, and this is the cell that earns the right to.

## 3. The shape of the tree

This is where a defect fixed on 2026-11-08 shows up as data. Plain k-means optimises
cohesion and says nothing about size. Because `recursive_split` recurses until every leaf
is under the cap, an uneven split is not absorbed — it is **amplified into depth**. One
9,932-series category came out at **depth 8 with nine leaves holding a single series
each**.

The fix is to keep the k-means centroids but cap assignment at `ceil(n/k)`, placing each
point most-confident-first. Measured on that category: depth 8 → 3, 1,105 names → 426,
nine singleton leaves → none, for two points of cohesion (0.953 → 0.931).

Two signatures tell you balancing is working: **derived depth stays shallow**, and
**there are no tiny leaves**.

In [5]:
with engine.connect() as c:
    print('derived depth (1 is the first level under a FRED category):')
    for r in c.execute(text('''
        SELECT depth, count(*) AS nodes, count(*) FILTER (WHERE is_leaf) AS leaves,
               sum(series_count) FILTER (WHERE is_leaf) AS series_in_leaves
          FROM fred_subcategory GROUP BY depth ORDER BY depth
    ''')):
        print(f'  depth {r[0]}: {r[1]:>6,} nodes  {r[2]:>6,} leaves  '
              f'{r[3] or 0:>8,} series in those leaves')

    print()
    print('children per interior node:')
    for r in c.execute(text('''
        WITH kids AS (
            SELECT parent_subcategory_id AS p, count(*) AS k FROM fred_subcategory
             WHERE parent_subcategory_id IS NOT NULL GROUP BY 1)
        SELECT min(k), round(avg(k), 1), percentile_disc(0.5) WITHIN GROUP (ORDER BY k),
               max(k) FROM kids
    ''')):
        print(f'  min={r[0]}  mean={r[1]}  median={r[2]}  max={r[3]}  (ceiling {fs.BRANCH_CAP})')

derived depth (1 is the first level under a FRED category):
  depth 1:    723 nodes       0 leaves         0 series in those leaves
  depth 2:  7,816 nodes   4,606 leaves   107,324 series in those leaves
  depth 3: 13,295 nodes  11,445 leaves   244,732 series in those leaves
  depth 4:  5,551 nodes   5,551 leaves   117,273 series in those leaves

children per interior node:


  min=2  mean=4.6  median=3  max=25  (ceiling 25)


In [6]:
with engine.connect() as c:
    rows = c.execute(text('''
        SELECT series_count, count(*) FROM fred_subcategory
         WHERE is_leaf GROUP BY 1 ORDER BY 1
    ''')).all()

widest = max(n for _, n in rows)
print(f'leaf sizes ({sum(n for _, n in rows):,} leaves, cap {fs.LEAF_CAP}):')
for size, n in rows:
    print(f'  {size:>3} series  {n:>6,}  {"#" * max(1, round(40 * n / widest))}')
print(f'\n  smallest leaf: {rows[0][0]} series   singleton leaves: '
      f'{dict(rows).get(1, 0)}')

leaf sizes (21,602 leaves, cap 25):
    8 series       1  #
   10 series       1  #
   11 series      10  #
   12 series       2  #
   13 series     306  ##
   14 series     423  ###
   15 series   1,525  ###########
   16 series     296  ##
   17 series     513  ####
   18 series     446  ###
   19 series     632  #####
   20 series   2,349  #################
   21 series   1,593  ############
   22 series   2,759  ####################
   23 series   1,543  ###########
   24 series   3,786  ############################
   25 series   5,417  ########################################

  smallest leaf: 8 series   singleton leaves: 0


The distribution leaning hard against the cap is the point. Leaves are *full*, so the
tree is as wide and shallow as the cap permits, and the smallest leaf is nowhere near 1.

This also corrected a cost model. `LEAF_OVERHEAD = 1.58` — nodes per leaf-floor — was
taken as a property of the problem; it was an artefact of unbalanced clustering. After
balancing it is **1.0**, and `modelled_nodes` simulates the recursion top-down instead of
multiplying a fudge factor by the leaf floor.

How much to trust it is worth measuring rather than asserting, because the projection is
what a run is authorised against. It is a *simulation of balanced splitting*, and the real
splitter sees real data: a cluster that happens to fall under the cap stops early, and `k`
is itself capped at the branching ceiling, so above 625 series the recursion takes a turn
the simple division does not.

In [7]:
with engine.connect() as c:
    print(f"{'category':>8} {'series':>8} {'predicted':>10} {'actual':>8}  path")
    for r in c.execute(text('''
        SELECT b.category_id, b.series_count, b.nodes, c.category_path
          FROM fred_subcategory_build b JOIN fred_category c USING (category_id)
         WHERE b.status = 'ok' ORDER BY b.series_count DESC NULLS LAST LIMIT 8
    ''')):
        print(f'{r[0]:>8} {r[1]:>8,} {fs.modelled_nodes(r[1]):>10,} {r[2]:>8,}  {r[3][:46]}')

    built = c.execute(text('''
        SELECT series_count, nodes FROM fred_subcategory_build WHERE status = 'ok'
    ''')).all()

exact = sum(1 for n, a in built if fs.modelled_nodes(n) == a)
under = sum(1 for n, a in built if fs.modelled_nodes(n) < a)
worst = max(built, key=lambda r: abs(fs.modelled_nodes(r[0]) - r[1]) / max(r[1], 1))
predicted, actual = sum(fs.modelled_nodes(n) for n, _ in built), sum(a for _, a in built)

print(f'\nacross {len(built)} built categories:')
print(f'  predicted exactly   : {exact} ({exact / len(built):.1%})')
print(f'  under-predicted     : {under}')
print(f'  worst single category: {worst[0]:,} series -> '
      f'predicted {fs.modelled_nodes(worst[0]):,}, actual {worst[1]:,}')
print(f'  total               : predicted {predicted:,} vs actual {actual:,} '
      f'({(predicted - actual) / actual:+.2%})')

category   series  predicted   actual  path
      97   53,187      3,151    3,127  Production & Business Activity > Housing
   32251   46,124      2,526    2,526  National Accounts > Flow of Funds
   33957   18,514      1,901    1,851  Production & Business Activity > Expenditures 
   33735    9,932        426      426  Population, Employment, & Labor Markets > Popu
   32255    9,707        426      425  Money, Banking, & Finance > Financial Indicato
   32349    9,633        426      425  Population, Employment, & Labor Markets > Prod
   32285    9,478        426      425  International Data > Indicators > Population, 
   33441    8,509        376      375  Production & Business Activity > Services

across 723 built categories:
  predicted exactly   : 675 (93.4%)
  under-predicted     : 8
  worst single category: 637 series -> predicted 50, actual 74
  total               : predicted 27,384 vs actual 27,385 (-0.00%)


So: **exact for 93% of categories and accurate to a fraction of a percent in aggregate**,
which is what a cost projection needs, but **not a per-category guarantee** — the worst
case is a third low, and the error goes in both directions. Budget a run from the total,
not from any one category, and treat a single category's estimate as indicative.

That asymmetry is the reason the figure quoted before this run was wrong twice over. The
earlier model counted the *leaf floor* rather than simulating the recursion, under-reporting
by a third, and per-name cost had been measured on sparse leaves; balanced leaves are full,
so more of the 18-title sample is sent. The two errors pointed in opposite directions and
very nearly cancelled, which is how `$4.29` came to agree with its predecessor and look
confirmed.

## 4. Browsing one tree by hand

The numbers above say the shape is right. They do not say the result is *legible*, and
that only reading it can settle. Pick a mid-sized category and walk it.

In [8]:
def show_tree(category_id, max_children=6, show_concepts=True):
    with engine.connect() as c:
        path = c.execute(text(
            'SELECT category_path FROM fred_category WHERE category_id = :i'),
            {'i': category_id}).scalar()
        rows = c.execute(text('''
            SELECT subcategory_id, parent_subcategory_id, depth, name, concepts,
                   series_count, is_leaf
              FROM fred_subcategory WHERE category_id = :i
             ORDER BY depth, subcategory_id
        '''), {'i': category_id}).all()

    print(f'[{category_id}] {path}\n')
    children = {}
    for r in rows:
        children.setdefault(r[1], []).append(r)

    def walk(parent, indent=''):
        kids = children.get(parent, [])
        for r in kids[:max_children]:
            mark = '*' if r[6] else '+'
            print(f'{indent}{mark} {r[3]}  ({r[5]} series)')
            if show_concepts and r[4]:
                print(f'{indent}    concepts: {r[4][:96]}')
            walk(r[0], indent + '    ')
        if len(kids) > max_children:
            print(f'{indent}  ... and {len(kids) - max_children} more siblings')

    walk(None)
    print('\n(* leaf, + interior)')


with engine.connect() as c:
    demo = c.execute(text('''
        SELECT category_id FROM fred_subcategory_build
         WHERE status = 'ok' AND series_count BETWEEN 200 AND 400
         ORDER BY series_count LIMIT 1
    ''')).scalar()

show_tree(demo)

[32846] International Data > Countries > Yemen, Republic of

+ Yemen Trade and Financial Sector Indicators  (201 series)
    concepts: Yemen, trade statistics, exports, banking, microfinance, financial inclusion, economic indicator
    * US State Exports to Yemen  (23 series)
        concepts: trade, shipments, merchandise trade, foreign trade, export firms, overseas sales, international 
    * US state exports to Yemen  (23 series)
        concepts: trade flows, foreign trade, international trade, shipments abroad, exporting firms, Middle East 
    * Identified US exporters to Yemen  (23 series)
        concepts: trade, export firms, foreign trade, merchandise exports, commerce, business counts, Middle East,
    * Yemen banking access and related indicators  (23 series)
        concepts: financial inclusion, access to finance, banking sector, bank penetration, human capital, educati
    * Microfinance and financial inclusion indicators  (17 series)
        concepts: financial inclusio

`concepts` is the column doing the real work, and it is deliberately *not* a summary of
the titles. It is generated alongside the name and told to include synonyms **absent**
from the member titles, which is what makes the structure reachable by ordinary
full-text search. Section 5 measures that; section 6 shows where it breaks.

## 5. The vocabulary bridge — and the operator that breaks it

The claim for this layer is that searching ~27,000 names and concept sets beats searching
846,628 titles, because a title says what a series *is called* and a concept set says what
it is *about*. Below, each query is run against both.

**The operator is not a detail.** `websearch_to_tsquery` ANDs its terms, which requires
one node's name-and-concepts to contain *every* word of the query. For the queries this
layer exists to answer, that returns nothing at all. Ranked OR matching is what makes it
work, and a search tool built on the obvious operator would make the whole layer look
useless.

In [9]:
QUERIES = [
    'steel making capacity',
    'heavy industry plant buildout',
    'coal and coke production',
    'household borrowing stress',
    'shipping container throughput',
]

AND_SQL = '''
    SELECT s.name FROM fred_subcategory s
     WHERE s.name_tsv @@ websearch_to_tsquery('english', :q) LIMIT 1
'''
OR_SQL = '''
    SELECT s.name, c.category_path, ts_rank(s.name_tsv, q.tsq) AS rank
      FROM fred_subcategory s JOIN fred_category c USING (category_id),
           (SELECT replace(plainto_tsquery('english', :q)::text, '&', '|')::tsquery AS tsq) q
     WHERE s.name_tsv @@ q.tsq ORDER BY rank DESC LIMIT 2
'''
TITLE_SQL = '''
    SELECT count(*) FROM fred_series
     WHERE to_tsvector('english', series_title) @@ plainto_tsquery('english', :q)
'''

with engine.connect() as c:
    for q in QUERIES:
        titles = c.execute(text(TITLE_SQL), {'q': q}).scalar()
        strict = c.execute(text(AND_SQL), {'q': q}).fetchone()
        print(f'\n"{q}"')
        print(f'  titles containing all terms : {titles}')
        print(f'  AND (websearch_to_tsquery)  : {strict[0] if strict else "-- nothing --"}')
        for r in c.execute(text(OR_SQL), {'q': q}):
            print(f'  OR, ranked {r[2]:.4f}         : {r[0]}')
            print(f'                                under {r[1][:60]}')


"steel making capacity"
  titles containing all terms : 0
  AND (websearch_to_tsquery)  : -- nothing --
  OR, ranked 0.0456         : Metals manufacturing capacity and output indexes
                                under Production & Business Activity > Industrial Production & Cap
  OR, ranked 0.0288         : Iron and steel industrial production series
                                under Production & Business Activity > Industrial Production & Cap



"heavy industry plant buildout"
  titles containing all terms : 0
  AND (websearch_to_tsquery)  : -- nothing --
  OR, ranked 0.0456         : Machinery manufacturing output indexes
                                under Production & Business Activity > Industrial Production & Cap
  OR, ranked 0.0374         : Nondurable goods manufacturing output by industry
                                under Production & Business Activity > Industrial Production & Cap



"coal and coke production"
  titles containing all terms : 7
  AND (websearch_to_tsquery)  : -- nothing --
  OR, ranked 0.0491         : Petroleum and coal products productivity measures
                                under Population, Employment, & Labor Markets > Productivity & Cos
  OR, ranked 0.0478         : Petroleum and coal industry production and cost measures
                                under Population, Employment, & Labor Markets > Productivity & Cos



"household borrowing stress"
  titles containing all terms : 0
  AND (websearch_to_tsquery)  : -- nothing --
  OR, ranked 0.0507         : Consumer lending and household borrowing
                                under Money, Banking, & Finance > Banking > Consumer Credit
  OR, ranked 0.0507         : Household financial stress and attitude surveys
                                under Money, Banking, & Finance > Financial Indicators > Consumer 



"shipping container throughput"
  titles containing all terms : 0
  AND (websearch_to_tsquery)  : -- nothing --
  OR, ranked 0.0253         : Rail, canal and shipping traffic
                                under Academic Data > NBER Macrohistory Database > Transportation 
  OR, ranked 0.0253         : Freight transport producer price indexes
                                under Prices > Producer Price Indexes (PPI) > Industry Based > Tra


Read the first line of each block against the last. Lexical search over the titles returns
**zero** rows for queries whose subject matter FRED covers in detail; the derived layer
puts a plausible node first. That gap is the entire justification for spending money on
naming.

## 6. Where the scheme fails: sibling name collisions

This is the finding this notebook was worth writing for.

A node is named from its own members, with **no knowledge of its siblings**. That is
cheap and parallel — 8 calls in flight, siblings being independent — and it is fine as
long as the clusters are genuinely about different things. When they are not, the namer
produces the same name several times over, and the result is a parent whose children
cannot be told apart.

In [10]:
with engine.connect() as c:
    r = c.execute(text('''
        WITH sib AS (
            SELECT category_id, parent_subcategory_id, lower(name) AS n, count(*) AS dupes
              FROM fred_subcategory GROUP BY 1, 2, 3 HAVING count(*) > 1)
        SELECT count(*), sum(dupes), count(DISTINCT category_id), max(dupes) FROM sib
    ''')).fetchone()
    total = c.execute(text('SELECT count(*) FROM fred_subcategory')).scalar()

print(f'sibling groups with a repeated name : {r[0]:,}')
print(f'nodes involved                      : {r[1]:,}  ({r[1] / total:.1%} of all nodes)')
print(f'categories affected                 : {r[2]:,}')
print(f'worst single collision              : {r[3]} identical siblings')

with engine.connect() as c:
    print('\nworst offenders:')
    for row in c.execute(text('''
        SELECT s.category_id, c.category_path, s.name, count(*) AS dupes
          FROM fred_subcategory s JOIN fred_category c USING (category_id)
         GROUP BY s.category_id, c.category_path, s.parent_subcategory_id, s.name
        HAVING count(*) > 1 ORDER BY 4 DESC LIMIT 8
    ''')):
        print(f'  {row[3]:>2}x  "{row[2]}"')
        print(f'        [{row[0]}] {row[1][:66]}')

sibling groups with a repeated name : 1,810
nodes involved                      : 4,445  (16.2% of all nodes)
categories affected                 : 366
worst single collision              : 11 identical siblings

worst offenders:
  11x  "PPP-converted GDP per capita by country"
        [33111] Academic Data > Penn World Table 7.1 > Real GDP per Capita (Consta
  11x  "Gender parity in education"
        [33526] International Data > Indicators > Population, Employment, & Labor 
  11x  "Consumer price index statistics"
        [33004] International Data > Indicators > Prices > Consumer Price Indexes 
   9x  "Metro area housing listing statistics"
        [97] Production & Business Activity > Housing
   9x  "International migrant stock by country"
        [33498] International Data > Indicators > Population, Employment, & Labor 
   9x  "Consumer price index statistics"
        [32266] International Data > Indicators > Prices > Consumer Price Indexes
   8x  "County-Level Educational Attainm

### Why, and why it is not a naming bug

Look at the members. The titles in these leaves are *identical except for a place name*,
and the place name sits at the end of a long string.

In [11]:
with engine.connect() as c:
    group = c.execute(text('''
        SELECT category_id, parent_subcategory_id, name, count(*) AS k
          FROM fred_subcategory WHERE is_leaf
         GROUP BY 1, 2, 3 HAVING count(*) > 2 ORDER BY 4 DESC LIMIT 1
    ''')).fetchone()
    print(f'{group[3]} sibling leaves all named "{group[2]}"')
    print('in', c.execute(text('SELECT category_path FROM fred_category WHERE category_id = :i'),
                          {'i': group[0]}).scalar(), '\n')
    leaves = [r[0] for r in c.execute(text('''
        SELECT subcategory_id FROM fred_subcategory
         WHERE category_id = :c AND name = :n AND is_leaf
           AND parent_subcategory_id IS NOT DISTINCT FROM :p
         ORDER BY subcategory_id LIMIT 3
    '''), {'c': group[0], 'n': group[2], 'p': group[1]})]
    for sid in leaves:
        print(f'  leaf {sid}:')
        for r in c.execute(text('''
            SELECT s.series_title FROM fred_series_subcategory ss
              JOIN fred_series s USING (series_id)
             WHERE ss.subcategory_id = :i ORDER BY s.series_title LIMIT 4
        '''), {'i': sid}):
            title = r[0]
            # Keep the TAIL of the title. The differentiator is the place name at the
            # end, so truncating from the left is what hides the finding.
            shown = title if len(title) <= 86 else title[:46] + ' ... ' + title[-35:]
            print(f'    {shown}')

11 sibling leaves all named "PPP-converted GDP per capita by country"
in Academic Data > Penn World Table 7.1 > Real GDP per Capita (Constant Prices) 

  leaf 5465:
    Purchasing Power Parity Converted GDP Per Capi ... s of domestic absorption for Angola
    Purchasing Power Parity Converted GDP Per Capi ... s of domestic absorption for Bhutan
    Purchasing Power Parity Converted GDP Per Capi ... s of domestic absorption for Brunei
    Purchasing Power Parity Converted GDP Per Capi ... of domestic absorption for Cambodia
  leaf 5466:
    Purchasing Power Parity Converted GDP Per Capi ...  Consumption, Investment for Bhutan
    Purchasing Power Parity Converted GDP Per Capi ...  Consumption, Investment for Brunei
    Purchasing Power Parity Converted GDP Per Capi ... onsumption, Investment for Cambodia
    Purchasing Power Parity Converted GDP Per Capi ... nt Consumption, Investment for Chad
  leaf 5468:
    Purchasing Power Parity Converted GDP Per Capita (Chain Series) for Belgium
 

These are **facet** categories: one measure across many places. The titles differ by
*place*, not by *measure*, so an embedding of the title is nearly constant across the
whole category and there is no semantic axis to cluster on. The splitter then does what it
is told — carve the category into equal parts — and every part is honestly described by
the same name.

So the collision is not the namer misbehaving. It is the **semantic scheme being applied
to a category that has no semantic variation**, and the name is the first place that
becomes visible.

This is the same argument that excludes `U.S. Regional Data` from this build (§4.4 of the
procedure). The correction worth recording: that half was excluded on the grounds that
regional titles differ by place rather than measure — and **the same structure is present
in the half that was built**, concentrated in `International Data` and the cross-country
academic datasets. The exclusion was right; treating it as a property of the regional
branch was not.

In [12]:
with engine.connect() as c:
    print(f"{'collided':>9} {'nodes':>8} {'share':>7}  branch")
    for r in c.execute(text('''
        WITH dup AS (
            SELECT category_id, parent_subcategory_id, name, count(*) AS k
              FROM fred_subcategory GROUP BY 1, 2, 3 HAVING count(*) > 1),
        per_branch AS (
            SELECT split_part(c.category_path, ' > ', 1) || ' > ' ||
                   split_part(c.category_path, ' > ', 2) AS branch,
                   count(*) AS nodes
              FROM fred_subcategory s JOIN fred_category c USING (category_id)
             GROUP BY 1),
        collided AS (
            SELECT split_part(c.category_path, ' > ', 1) || ' > ' ||
                   split_part(c.category_path, ' > ', 2) AS branch,
                   sum(dup.k) AS dupes
              FROM dup JOIN fred_category c USING (category_id)
             GROUP BY 1)
        SELECT collided.dupes, per_branch.nodes,
               round(100.0 * collided.dupes / per_branch.nodes, 1), per_branch.branch
          FROM collided JOIN per_branch USING (branch)
         ORDER BY collided.dupes DESC LIMIT 10
    ''')):
        print(f'{r[0]:>9,} {r[1]:>8,} {r[2]:>6}%  {r[3][:56]}')

 collided    nodes   share  branch
    1,277    5,328   24.0%  International Data > Indicators
      458    3,143   14.6%  Production & Business Activity > Housing
      418    5,440    7.7%  International Data > Countries
      294    1,851   15.9%  Production & Business Activity > Expenditures
      193      604   32.0%  Population, Employment, & Labor Markets > Population
      171    2,644    6.5%  National Accounts > Flow of Funds
      156      633   24.6%  Prices > Producer Price Indexes (PPI)
      132      298   44.3%  Academic Data > Penn World Table 7.1
      116    1,017   11.4%  Population, Employment, & Labor Markets > Productivity &
      111      890   12.5%  National Accounts > National Income & Product Accounts


A facet category needs a facet split — group by the place named in the title, which is
data FRED already has and the embedding throws away — not a semantic one. That is a
separate pass, and until it exists these categories are split correctly and labelled
uselessly.

## 7. How many clicks to a leaf

Depth is **two stacked trees**: FRED's own `category_path`, then the derived levels. The
two are anti-correlated, which is why the branching ceiling could be tightened without
making browsing worse — the categories holding the most series sit shallowest in FRED's
tree, so the derived levels are added where there is room for them.

In [13]:
with engine.connect() as c:
    rows = c.execute(text('''
        WITH leaf AS (
            SELECT array_length(string_to_array(c.category_path, ' > '), 1) AS fred,
                   s.depth AS derived
              FROM fred_subcategory s JOIN fred_category c USING (category_id)
             WHERE s.is_leaf)
        SELECT fred + derived AS total, count(*) FROM leaf GROUP BY 1 ORDER BY 1
    ''')).all()
    stats = c.execute(text('''
        WITH leaf AS (
            SELECT s.depth + array_length(string_to_array(c.category_path, ' > '), 1) AS t
              FROM fred_subcategory s JOIN fred_category c USING (category_id)
             WHERE s.is_leaf)
        SELECT percentile_disc(0.5) WITHIN GROUP (ORDER BY t),
               percentile_disc(0.95) WITHIN GROUP (ORDER BY t), max(t) FROM leaf
    ''')).fetchone()

widest = max(n for _, n in rows)
print('total levels from the FRED root to a leaf:')
for total, n in rows:
    print(f'  {total:>2} {n:>7,}  {"#" * max(1, round(44 * n / widest))}')
print(f'\n  median={stats[0]}  p95={stats[1]}  max={stats[2]}')

total levels from the FRED root to a leaf:
   3       2  #
   4     328  #
   5   4,545  ##################
   6  11,214  ############################################
   7   4,165  ################
   8   1,176  #####
   9     172  #

  median=6  p95=8  max=9


The derived levels are the cheap half of that. The expensive half is FRED's own tree, and
the levers on it are structural rather than semantic: **pass-through** categories with a
single child add a click and no choice, and **dead** categories hold no series at any
depth. Both are reported here and neither is acted on by this notebook.

In [14]:
with engine.connect() as c:
    passthrough = c.execute(text('''
        SELECT count(*) FROM (
            SELECT parent_id FROM fred_category
             WHERE parent_id IS NOT NULL GROUP BY parent_id HAVING count(*) = 1) t
    ''')).scalar()
    dead = c.execute(text('''
        SELECT count(*) FROM fred_category c
         WHERE NOT EXISTS (SELECT 1 FROM fred_series_category WHERE category_id = c.category_id)
           AND NOT EXISTS (SELECT 1 FROM fred_category k WHERE k.parent_id = c.category_id)
    ''')).scalar()
print(f'single-child (pass-through) parents : {passthrough:,}')
print(f'categories with no series and no children : {dead:,}')

single-child (pass-through) parents : 91
categories with no series and no children : 74


## 8. What is incomplete

Two of these are deliberate and two are faults. Keeping them in one place is the point —
an unrecorded decision is indistinguishable from a bug.

In [15]:
with engine.connect() as c:
    scoped = {name: c.execute(text(f'''
        SELECT count(*) FROM fred_category c
         WHERE c.excluded_reason IS NULL {clause}
    ''')).scalar() for name, clause in fs.SCOPES.items()}
    print('catalogue by scope:', scoped)

    built = c.execute(text('''
        SELECT count(DISTINCT category_id) FROM fred_subcategory_build
         WHERE status IN ('ok', 'skipped')
    ''')).scalar()
    print(f'categories with a finished build : {built:,}')
    print(f'still pending in non-regional    : {len(fs.pending_builds(engine)):,}')
    print(f'pending in regional              : '
          f'{len(fs.pending_builds(engine, scope="regional")):,}   <- deliberate, see section 6')

catalogue by scope: {'all': 5201, 'non-regional': 1028, 'regional': 4173}
categories with a finished build : 924
still pending in non-regional    : 0


pending in regional              : 3,995   <- deliberate, see section 6


In [16]:
# A fault, not a decision: naming failed and the node carries a placeholder.
unnamed = fs.unnamed_nodes(engine)
print(f'unnamed nodes: {len(unnamed)}')
with engine.connect() as c:
    for r in c.execute(text('''
        SELECT s.category_id, c.category_path, s.depth, s.series_count, s.is_leaf,
               (SELECT count(*) FROM fred_subcategory k
                 WHERE k.parent_subcategory_id = s.subcategory_id) AS children
          FROM fred_subcategory s JOIN fred_category c USING (category_id)
         WHERE s.name LIKE 'UNNAMED%' ORDER BY s.depth, s.series_count DESC LIMIT 10
    ''')):
        kind = 'leaf' if r[4] else f'interior, {r[5]} children'
        print(f'  depth {r[2]}  {r[3]:>5} series  ({kind})  [{r[0]}] {r[1][:46]}')

unnamed nodes: 38
  depth 1     56 series  (interior, 3 children)  [32725] International Data > Countries > Holy See (Vat
  depth 1     42 series  (interior, 2 children)  [32681] International Data > Countries > Cocos (Keelin
  depth 1     38 series  (interior, 2 children)  [32789] International Data > Countries > Pitcairn
  depth 2    171 series  (interior, 7 children)  [7] National Accounts > U.S. Trade & International
  depth 2     72 series  (interior, 3 children)  [32427] International Data > Indicators > Prices > Pro
  depth 2     60 series  (interior, 3 children)  [33835] Academic Data > Banking and Monetary Statistic
  depth 2     41 series  (interior, 2 children)  [32344] International Data > Countries > Turkey
  depth 2     41 series  (interior, 2 children)  [32343] International Data > Countries > Slovenia
  depth 2     41 series  (interior, 2 children)  [32276] International Data > Countries > Norway
  depth 2     40 series  (interior, 2 children)  [32282] International Dat

An unnamed node is recorded three ways on purpose — the placeholder name,
`fred_subcategory_build.naming_failures`, and a warning in the run log — because the
original failure mode was worse than the failure: `UNNAMED` went into a browsable tree
beside a build row reading `ok`, so nothing distinguished a complete category from one
with a hole in it.

A **depth-1** unnamed node is the bad case. That is a category's own root, so the top of
its derived tree has no label at all. Rebuild a single category without touching the rest:

```bash
python clients/bin/fred_subcategories.py build --category 7
```

In [17]:
# The other fault, inherited from collection: series that belong to no category at all.
# 1,000 retired ALLQ* series were orphaned when category 34111's surplus memberships were
# pruned. The rows were kept deliberately -- one INSERT restores them -- but nothing yet
# decides whether they should be deleted or re-attached.
with engine.connect() as c:
    orphans = c.execute(text('''
        SELECT count(*) FROM fred_series s
         WHERE NOT EXISTS (SELECT 1 FROM fred_series_category WHERE series_id = s.series_id)
    ''')).scalar()
print(f'series in no category (so reachable by no derived leaf): {orphans:,}')

series in no category (so reachable by no derived leaf): 1,000


## 9. Running it outside a notebook

```bash
python clients/bin/fred_subcategories.py status                     # read-only
python clients/bin/fred_subcategories.py verify                     # exit 1 on any violation
python clients/bin/fred_subcategories.py build --dry-run            # cost and time first
python clients/bin/fred_subcategories.py build --scope non-regional # the run this examines
python clients/bin/fred_subcategories.py build --category 7         # one category
python clients/bin/fred_subcategories.py resync                     # after FRED changes
```

`verify` is the one to wire into anything automated: it asserts both caps and the absence
of unnamed nodes, and returns a non-zero exit status rather than printing a warning
nobody reads.

**A second run costs nothing.** `pending_builds` skips any category already `ok` or
`skipped` under the current prompt *and* split version, so an interrupted run is resumed
by invoking it again and a finished one makes no calls. That was load-bearing during this
build: 18 categories from an earlier proving run were correctly skipped, so the 906
pending categories were named exactly once.